# SKY130 FET characterization — I–V families, $g_m/I_D$, and $f_T$

Companion to web-app testbenches **04–07** (`SKY130 NFET/PFET output curves`,
`SKY130 NFET/PFET f_T`). The devices are the *real* `sky130_fd_pr__*` BSIM4.8
models evaluated by the server — nothing here is a hand-fit — so the numbers
this notebook extracts are the PDK's own physics.

The workflow every notebook in this folder follows:

1. pull a testbench from the running web app (`Session.load_example`),
2. run analyses **server-side** (same engine as the browser's Run button),
3. reduce and compare against theory in numpy, right here.

Requires the web app running locally (`python webapp/server.py`, or point
`PHOTONFLUX_URL` elsewhere). Nothing below touches your canvas: schematics
are passed explicitly to each run, so the browser mirror is left alone.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session, si

s = Session()
s

## 1. Output curves ($I_D$–$V_{DS}$, stepped $V_{GS}$)

Testbench 04 puts four NFET flavors (`01v8`, `lvt`, `5v`, `nvt`) on shared
gate/drain rails `VG1`/`VD1`. Each source goes to ground through a **1 Ω
sense resistor**, so the probed source-node voltage *is* the drain current
in amps — the schematic-level equivalent of a SMU's ammeter. Testbench 05
is the PFET twin with everything mirrored negative.

`s.run(schematic=bench)` with no analysis argument runs the analysis stored
in the example — exactly what the browser's Run button does.

In [ ]:
nfet_bench = s.load_example("04_sky130_nfet_output_curves")
pfet_bench = s.load_example("05_sky130_pfet_output_curves")

out_n = s.run(schematic=nfet_bench)
out_p = s.run(schematic=pfet_bench)
print(out_n.log[-1])
print(out_n.names)

In [ ]:
NFLAV = ["id_01v8", "id_lvt", "id_5v", "id_nvt"]
PFLAV = ["id_01v8", "id_lvt", "id_hvt", "id_5v"]

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex="row")
for ax, probe in zip(axes[0], NFLAV):
    for name, idv in out_n.family(probe).items():
        ax.plot(out_n.x, np.abs(idv) * 1e3,
                label=f"Vgs={name.split('@')[1].strip()}")
    ax.set_title(f"nfet {probe[3:]}"), ax.grid(alpha=.3)
    ax.set_xlabel("Vds [V]")
for ax, probe in zip(axes[1], PFLAV):
    for name, idv in out_p.family(probe).items():
        ax.plot(-out_p.x, np.abs(idv) * 1e3,
                label=f"Vgs={name.split('@')[1].strip()}")
    ax.set_title(f"pfet {probe[3:]}"), ax.grid(alpha=.3)
    ax.set_xlabel("|Vds| [V]")
axes[0][0].set_ylabel("Id [mA]  (W=1 µm)")
axes[1][0].set_ylabel("|Id| [mA]  (W=2 µm)")
axes[0][0].legend(fontsize=8), axes[1][0].legend(fontsize=8)
fig.tight_layout()

## 2. Transfer curves and parameter extraction

Same benches, our own analysis: hold $V_{DS}$ at the rail and sweep the
shared gate. From one $I_D(V_{GS})$ curve per flavor we extract

* $g_m = dI_D/dV_{GS}$ (numerical gradient),
* $V_{th}$ by **max-$g_m$ extrapolation**: at the steepest point,
  $V_{th} = V_{GS}^\* - I_D^\*/g_m^\*$,
* **subthreshold swing** $SS = dV_{GS}/d(\log_{10} I_D)$, fit over the
  exponential decade(s) — the ~60 mV/dec-limited steepness,
* the $g_m/I_D$ curve — transconductance efficiency, the workhorse of
  modern analog sizing (used again in notebook 05 for the diff pair).

In [ ]:
tr_n = s.dcsweep("VG1", "V", 0, 1.8, points=181, schematic=nfet_bench)
tr_p = s.dcsweep("VG1", "V", 0, -1.8, points=181, schematic=pfet_bench)


def extract(vg, idr):
    """Vth (max-gm extrapolation), SS, and gm from one transfer curve."""
    gm = np.gradient(idr, vg)
    k = int(np.argmax(gm))
    vth = float(vg[k] - idr[k] / gm[k])
    sub = (idr > 1e-11) & (idr < 1e-7)          # the exponential region
    ss = (float(np.polyfit(np.log10(idr[sub]), vg[sub], 1)[0] * 1e3)
          if sub.sum() > 5 else float("nan"))    # mV/dec; nan for native Vth~0
    return {"vth": vth, "ss_mv_dec": ss, "gm": gm}


nfet, pfet = {}, {}
for probe in NFLAV:
    idr = np.abs(tr_n[probe])
    nfet[probe[3:]] = {"vgs": tr_n.x, "id": idr, **extract(tr_n.x, idr)}
for probe in PFLAV:
    vsg, idr = -tr_p.x, np.abs(tr_p[probe])     # report the PFET in |V|, |I|
    pfet[probe[3:]] = {"vgs": vsg, "id": idr, **extract(vsg, idr)}

print(f"{'flavor':10s} {'Vth [V]':>9s} {'SS [mV/dec]':>12s} {'Id@rail [µA/µm]':>16s}")
for fam, dev, w in [(nfet, "nfet", 1.0), (pfet, "pfet", 2.0)]:
    for k, d in fam.items():
        print(f"{dev} {k:5s} {d['vth']:9.3f} {d['ss_mv_dec']:12.1f} "
          f"{d['id'][-1] / w * 1e6:16.1f}")

In [ ]:
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 3.6))
for k, d in nfet.items():
    a1.plot(d["vgs"], d["id"] * 1e3, label=f"nfet {k}")
    a2.semilogy(d["vgs"], np.maximum(d["id"], 1e-14), label=f"nfet {k}")
    a3.plot(d["vgs"], d["gm"] * 1e3, label=f"nfet {k}")
    a3.axvline(d["vth"], color=a3.lines[-1].get_color(), ls=":", lw=.8)
a1.set_ylabel("Id [mA]"), a2.set_ylabel("Id [A] (log)")
a3.set_ylabel("gm [mS]  (dotted: extracted Vth)")
for a in (a1, a2, a3):
    a.set_xlabel("Vgs [V]"), a.grid(alpha=.3), a.legend(fontsize=8)
fig.tight_layout()

The log plot is the story: the `nvt` (native) device barely turns off —
its $V_{th}\approx 0$ is *by construction* (no channel implant), which is
why it appears in cascode/source-follower roles, never as a switch. The
`5v` device pays for its thick oxide with the highest $V_{th}$ and the
softest $g_m$.

## 3. The $g_m/I_D$ design chart

Plotting transconductance efficiency against **current density** $I_D/W$
makes the curve geometry-independent — one lookup chart per flavor sizes
any transistor. Weak inversion saturates near the bipolar-like limit
$g_m/I_D \to 1/(n V_T) \approx 25\text{–}30\ \mathrm{V^{-1}}$; strong
inversion trades efficiency for speed as $g_m/I_D \approx 2/V_{ov}$.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for fam, dev, w, ax in [(nfet, "nfet", 1.0, a1), (pfet, "pfet", 2.0, a2)]:
    for k, d in fam.items():
        dens = d["id"] / w * 1e6                 # µA/µm
        eff = np.divide(d["gm"], d["id"], out=np.zeros_like(d["gm"]),
                        where=d["id"] > 1e-12)
        m = dens > 1e-3
        ax.semilogx(dens[m], eff[m], label=f"{dev} {k}")
    ax.axhline(1 / 0.026, color="gray", ls="--", lw=.8)
    ax.text(2e-3, 1 / 0.026 + 1, "1/VT (n=1) weak-inversion limit",
            fontsize=7, color="gray")
    ax.set_xlabel("Id / W  [µA/µm]"), ax.set_ylabel("gm/Id  [1/V]")
    ax.grid(alpha=.3, which="both"), ax.legend(fontsize=8)
fig.tight_layout()

## 4. Output resistance

Back to the output curves: in saturation $I_D \approx I_{D0}(1+\lambda
V_{DS})$, so a linear fit over the flat region gives
$r_o = 1/(dI_D/dV_{DS})$ and $\lambda = g_{ds}/I_D$. Short 0.15 µm
channels are leaky ($\lambda$ large, $r_o$ a few kΩ·µm) — the reason
notebook 05's diff pair needs its 4 kΩ loads, not $r_o$, to set gain.

In [ ]:
sat = out_n.x >= 1.2
print(f"{'nfet 01v8':12s} {'Id@1.5V':>12s} {'ro':>10s} {'lambda':>9s} {'gm*ro':>7s}")
ro_01v8 = {}
for name, idv in out_n.family("id_01v8").items():
    vgs = float(name.split("@")[1].strip().rstrip("V"))
    gds, i0 = np.polyfit(out_n.x[sat], idv[sat], 1)
    id15 = i0 + gds * 1.5
    ro = 1 / gds
    gm15 = float(np.interp(vgs, tr_n.x, nfet["01v8"]["gm"]))
    ro_01v8[vgs] = {"ro_ohm": float(ro), "lambda": float(gds / id15),
                    "id_a": float(id15)}
    print(f"  Vgs={vgs:4.2f} {id15 * 1e6:10.1f} µA {ro / 1e3:8.2f} kΩ "
          f"{gds / id15:9.3f} {gm15 * ro:7.1f}")

## 5. Transit frequency $f_T$

Testbenches 06/07 measure the **fixture-free $|h_{21}|$**: the server
drives the gate's bias source with a 1 V AC tone and takes the ratio of
drain to gate current — no 50 Ω pads, exactly the quantity whose 0 dB
crossing defines $f_T = g_m/(2\pi C_{gg})$. We re-extract the crossing
ourselves (log-interpolated) and check it against the server's log line.

In [ ]:
ftb_n = s.load_example("06_sky130_nfet_ft")
ftb_p = s.load_example("07_sky130_pfet_ft")
AC = dict(mode="ac", f_start=1e7, f_stop=1e12, points=121, z0=50)


def ft_from_h21(f, db):
    """First downward 0 dB crossing of |h21|, log-interpolated."""
    above = db > 0
    for k in range(1, len(f)):
        if above[k - 1] and not above[k]:
            d0, d1 = db[k - 1], db[k]
            t = d0 / (d0 - d1)
            return float(np.exp(np.log(f[k - 1])
                                + t * (np.log(f[k]) - np.log(f[k - 1]))))
    return float("nan")


ac_n = s.run(dict(AC), schematic=ftb_n)
ac_p = s.run(dict(AC), schematic=ftb_p)
fig, ax = plt.subplots(figsize=(7, 3.6))
for res, lbl in [(ac_n, "nfet 01v8 (W=1)"), (ac_p, "pfet 01v8 (W=2)")]:
    db = next(iter(res.family("h21").values()))
    ft = ft_from_h21(res.x, db)
    ax.semilogx(res.x, db, label=f"{lbl}: f_T = {ft / 1e9:.1f} GHz")
    ax.plot([ft], [0], "kx")
    print(f"{lbl}: notebook {ft / 1e9:.2f} GHz | server: "
          + next(l for l in res.log if "f_T" in l))
ax.axhline(0, color="gray", lw=.8)
ax.set_xlabel("frequency [Hz]"), ax.set_ylabel("|h21| [dB]")
ax.grid(alpha=.3, which="both"), ax.legend(fontsize=8)
fig.tight_layout()

### $f_T$ vs. bias — the speed/efficiency trade

One AC run per gate bias (the server's AC parameter sweep) maps $f_T$
across inversion. Plotted against the current density from section 2,
this is the designer's other half: $g_m/I_D$ falls with density while
$f_T$ rises — every amplifier bias point is a spot on these two curves.

In [ ]:
VGS_SWEEP = [0.7, 0.9, 1.1, 1.3, 1.5, 1.8]
sw = s.run({**AC, "points": 61, "sweep_instance": "VG1", "sweep_param": "V",
            "sweep_values": VGS_SWEEP}, schematic=ftb_n)

ft_bias = {}
for nm, db in sw.family("h21").items():
    vgs = si(nm.split("=")[1].split("(")[0].strip().rstrip("V"))
    ft_bias[vgs] = ft_from_h21(sw.x, np.asarray(db))

vgs_ax = np.array(sorted(ft_bias))
fts = np.array([ft_bias[v] for v in vgs_ax])
dens = np.interp(vgs_ax, tr_n.x, nfet["01v8"]["id"]) / 1.0 * 1e6   # µA/µm
effs = np.interp(vgs_ax, tr_n.x,
                 nfet["01v8"]["gm"] / np.maximum(nfet["01v8"]["id"], 1e-12))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(dens, fts / 1e9, "o-", color="tab:red")
ax.set_xlabel("Id / W  [µA/µm]"), ax.set_ylabel("f_T [GHz]", color="tab:red")
ax2 = ax.twinx()
ax2.semilogx(dens, effs, "s--", color="tab:blue")
ax2.set_ylabel("gm/Id [1/V]", color="tab:blue")
for v, d, f in zip(vgs_ax, dens, fts):
    ax.annotate(f"{v:g} V", (d, f / 1e9), fontsize=7,
                textcoords="offset points", xytext=(4, -8))
ax.grid(alpha=.3, which="both"), ax.set_title("nfet 01v8, W=1 µm, L=0.15 µm")
fig.tight_layout()

## 6. Save the extraction

Notebook 05 (differential pair) and 07 (ring oscillator) reuse these
curves instead of re-deriving them — the notebook version of a PDK
characterization report.

In [ ]:
Path("out").mkdir(exist_ok=True)
blob = {"w_um": {"nfet": 1.0, "pfet": 2.0}, "vds_v": 1.8,
        "ft_vs_vgs_nfet_01v8": {f"{v:g}": f for v, f in ft_bias.items()}}
for fam, dev in [(nfet, "nfet"), (pfet, "pfet")]:
    for k, d in fam.items():
        blob[f"{dev}_{k}"] = {
            "vgs": np.round(d["vgs"], 6).tolist(),
            "id": d["id"].tolist(), "gm": d["gm"].tolist(),
            "vth": d["vth"], "ss_mv_dec": d["ss_mv_dec"]}
blob["ro_01v8_nfet"] = ro_01v8
with open("out/fet_extraction.json", "w") as fh:
    json.dump(blob, fh)
print("saved out/fet_extraction.json",
      f"({Path('out/fet_extraction.json').stat().st_size / 1e3:.0f} kB)")

# sanity: the numbers notebook 05 will lean on
d = nfet["01v8"]
assert 0.3 < d["vth"] < 1.0, d["vth"]
assert 60 <= min(nfet["01v8"]["ss_mv_dec"], nfet["lvt"]["ss_mv_dec"]) < 120
assert fts[-1] > 20e9, "f_T at strong inversion should be tens of GHz"
print(f"nfet 01v8: Vth = {d['vth']:.3f} V, SS = {d['ss_mv_dec']:.0f} mV/dec, "
      f"peak f_T = {np.nanmax(fts) / 1e9:.0f} GHz")

---
**Where to go from here**

* Open testbench 04 in the browser and edit `w_um`/`l_um` — the mirror
  keeps this notebook's `s.pull()` in sync if you want to re-extract a
  custom geometry (`s.run()` with no arguments runs the live canvas).
* Notebook **05** sizes the differential pair from this extraction and
  checks the pen-and-paper answer against testbench 23.
* Notebook **07** does the same for digital timing with the square-law
  devices of testbenches 26/29.